# 01 — Data Loading & Preprocessing

This notebook covers:
1. Loading a public scRNA-seq dataset
2. Calculating QC metrics
3. Visualizing QC and choosing filter thresholds
4. Filtering, normalizing, and selecting highly variable genes
5. Running PCA
6. Saving the preprocessed object

**Dataset used for testing:** PBMC 3k (10x Genomics)  
Built into scanpy — no download needed. Later swap for real brain organoid data.

In [ ]:
import sys
sys.path.insert(0, '..')  # so Python can find the src/ package

import scanpy as sc
from src import preprocessing as pp
from src import visualization as viz
from src import utils

# scanpy settings
sc.settings.verbosity = 1          # 0=errors only, 3=very verbose
sc.settings.set_figure_params(dpi=100, facecolor='white')

utils.ensure_dirs()
print('Setup complete.')

## 1. Load Dataset

We use `pbmc3k` (Peripheral Blood Mononuclear Cells, 2,700 cells) as a **stand-in**  
to verify the pipeline works before using real brain organoid data.

When you have your real data, replace this cell with:
```python
adata = utils.load_h5ad('your_file.h5ad')
# or
adata = utils.load_10x_mtx('your_folder/')
```

In [ ]:
# Load the built-in PBMC 3k dataset (downloads ~6 MB on first run)
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()

utils.summarize(adata, 'PBMC 3k (raw)')

## 2. QC Metrics

In [ ]:
adata = pp.calculate_qc_metrics(adata)

In [ ]:
# Violin plots — look for outliers on the high end
viz.plot_qc_metrics(adata)

In [ ]:
# Scatter plots — cells top-right of the count/gene cloud may be doublets
viz.plot_qc_scatter(adata)

## 3. Filter Cells and Genes

Typical thresholds (adjust based on your QC plots above):
- `min_genes=200` — remove empty droplets
- `max_genes=2500` — remove doublets (for PBMC; use ~6000 for brain organoids)
- `max_pct_mt=5` — remove dead/damaged cells (PBMC is strict; brain organoids can be up to 20%)
- `min_cells=3` — remove genes detected in fewer than 3 cells

In [ ]:
adata = pp.filter_cells_and_genes(
    adata,
    min_genes=200,
    max_genes=2500,
    max_pct_mt=5.0,
    min_cells=3,
)

utils.summarize(adata, 'After filtering')

## 4. Normalize and Log-Transform

In [ ]:
adata = pp.normalize_and_log(adata)

## 5. Highly Variable Genes

In [ ]:
adata = pp.select_highly_variable_genes(adata, n_top_genes=2000)
viz.plot_hvg(adata)

## 6. PCA

In [ ]:
adata = pp.run_pca(adata, n_comps=50)

# Scree plot — pick the number of PCs at the 'elbow'
viz.plot_pca_variance(adata, n_pcs=30)

## 7. Save Preprocessed Data

In [ ]:
utils.save_processed(adata, 'pbmc3k_preprocessed.h5ad')
print('Done. Move on to 02_umap_clustering.ipynb')